# ComplexSineSynth Model Benchmark

This notebook evaluates the trained DDSP VAE model with ComplexSineSynth:
- Input vs output audio comparison (waveform + spectrogram)
- Per-chunk reconstruction metrics
- Latent space visualization (PCA/t-SNE)
- Latent interpolation between chunks
- Random latent generation

In [1]:
import sys
sys.path.insert(0, '/home/btadeusz/code/ddsp_vae')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
from math import ceil

from ddsp import DDSP
from ddsp.audio_feature_dataset import AudioFeatureDataset
from ddsp.interfaces import build_control_space
from omegaconf import OmegaConf

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


## 1. Load Model & Dataset

In [2]:
# Config
cfg = OmegaConf.load('/home/btadeusz/code/ddsp_vae/configs/complex_sine/experiment_complex_sine_single_1024_frame.yaml')
fs = int(cfg.audio.fs)
resampling_factor = int(cfg.model.resampling_factor)
n_signal = int(fs * float(cfg.audio.chunk_duration_s))
control_space = build_control_space(cfg.model.control_space)

print(f'Sample rate: {fs} Hz')
print(f'Chunk duration: {cfg.audio.chunk_duration_s}s ({n_signal} samples)')
print(f'Resampling factor: {resampling_factor}')
print(f'Control frames per chunk: {ceil(n_signal / resampling_factor)}')

Sample rate: 44100 Hz
Chunk duration: 2.0s (88200 samples)
Resampling factor: 1024
Control frames per chunk: 87


In [3]:
# Dataset
dataset = AudioFeatureDataset(
    dataset_path=cfg.data.dataset_path,
    n_signal=n_signal,
    sampling_rate=fs,
    resampling_factor=resampling_factor,
    control_space=control_space,
)
print(f'Dataset chunks: {len(dataset)}')

Loading from cache: /mnt/mariadata/datasets/surrogate/single_sound/audio_cache_16d6097f.lmdb
Dataset chunks: 30


/home/btadeusz/code/ddsp_vae/ddsp/audio_feature_dataset.py:192: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  audio_cpu[pos:pos+n_samps] = torch.from_numpy(a_arr)


In [5]:
# Load model from best checkpoint
from ddsp.utils import find_checkpoint

ckpt_dir = '/home/btadeusz/code/ddsp_vae/training/synth/complex_sine_single_1024_frame'
ckpt_path = find_checkpoint(ckpt_dir, typ='best')
print(f'Loading: {ckpt_path}')

ddsp = DDSP.load_from_checkpoint(
  ckpt_path,
  map_location=device,
  control_space=control_space,
)
ddsp = ddsp.to(device)
ddsp.eval()

n_params = sum(p.numel() for p in ddsp.parameters())
print(f'Model parameters: {n_params:,}')
print(f'Latent size: {ddsp.latent_size}')
print(f'Synths: {[type(s).__name__ for s in ddsp.synths]}')

Loading: /home/btadeusz/code/ddsp_vae/training/synth/complex_sine_single_1024_frame/best-v31.ckpt
Building synthesizers..., resampling_factor: 1024
Model parameters: 17,301,916
Latent size: 512
Synths: ['ComplexSineSynth']


/home/btadeusz/miniconda3/envs/ddsp/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


## 2. Encode-Decode All Chunks

In [ ]:
@torch.no_grad()
def encode_chunk(ddsp, audio):
    """Encode audio → (mu, logvar, z). audio: [n_signal] or [B, n_signal]."""
    if audio.dim() == 1:
        audio = audio.unsqueeze(0)
    audio = audio.to(device)
    mu, logvar = ddsp.encoder(audio)  # [B, T_ctl, latent_size]
    z, _ = ddsp.encoder.reparametrize(mu, logvar)
    z = ddsp._smooth_latents(z)
    return mu, logvar, z

@torch.no_grad()
def decode_from_z(ddsp, z, audio=None, features=None):
    """Decode latent z → audio. z: [B, T_ctl, latent_size]."""
    B, T_ctl = z.shape[0], z.shape[1]
    if features is None:
        features = torch.zeros(B, T_ctl, max(ddsp.feature_dim, 1), device=z.device)
    features_for_dec = features if ddsp.feature_dim > 0 else torch.zeros(B, T_ctl, 1, device=z.device)
    T_min = min(features_for_dec.shape[1], z.shape[1])
    synth_params = ddsp.decoder(features_for_dec[:, :T_min], z[:, :T_min],
                                audio=audio, hop_size=ddsp.resampling_factor)
    signal = ddsp._synthesize(synth_params)
    return signal

@torch.no_grad()
def autoencode_chunk(ddsp, audio, features):
    """Full encode→decode pipeline. Returns (y_audio, mu, z)."""
    if audio.dim() == 1:
        audio = audio.unsqueeze(0)
    if features.dim() == 2:
        features = features.unsqueeze(0)
    audio = audio.to(device)
    features = features.to(device)
    y_audio = ddsp(audio, features)
    mu, logvar, z = encode_chunk(ddsp, audio)
    return y_audio, mu, z

print('Helper functions ready.')

In [ ]:
# Run autoencoding on all chunks
all_mu = []
all_z = []
all_x = []
all_y = []
all_mse = []

for i in range(len(dataset)):
    audio, features, idx = dataset[i]
    y_audio, mu, z = autoencode_chunk(ddsp, audio, features)

    x_np = audio.cpu().numpy()
    y_np = y_audio.squeeze().cpu().numpy()
    min_len = min(len(x_np), len(y_np))
    mse = np.mean((x_np[:min_len] - y_np[:min_len])**2)

    all_mu.append(mu.squeeze(0).cpu())
    all_z.append(z.squeeze(0).cpu())
    all_x.append(x_np)
    all_y.append(y_np)
    all_mse.append(mse)

all_mu_cat = torch.stack(all_mu)   # [N, T_ctl, latent_size]
all_z_cat = torch.stack(all_z)     # [N, T_ctl, latent_size]

print(f'Processed {len(dataset)} chunks')
print(f'Latent shape per chunk: {all_mu[0].shape}')
print(f'MSE — mean: {np.mean(all_mse):.6f}, min: {np.min(all_mse):.6f}, max: {np.max(all_mse):.6f}')

## 3. Input vs Output — Audio Comparison

In [ ]:
def plot_comparison(x, y, fs, title='', chunk_idx=0):
    """Plot waveform and spectrogram comparison for one chunk."""
    min_len = min(len(x), len(y))
    x, y = x[:min_len], y[:min_len]
    t = np.arange(min_len) / fs
    mse = np.mean((x - y)**2)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    fig.suptitle(f'{title}Chunk {chunk_idx} — MSE: {mse:.6f}', fontsize=13)

    # Waveforms
    axes[0].plot(t, x, alpha=0.7, label='Input', linewidth=0.5)
    axes[0].plot(t, y, alpha=0.7, label='Output', linewidth=0.5)
    axes[0].set_ylabel('Amplitude')
    axes[0].legend(loc='upper right')
    axes[0].set_title('Waveform')

    # Spectrograms
    for ax, data, label in [(axes[1], x, 'Input'), (axes[2], y, 'Output')]:
        ax.specgram(data, Fs=fs, NFFT=2048, noverlap=1024, cmap='magma')
        ax.set_ylabel('Freq (Hz)')
        ax.set_title(f'{label} Spectrogram')
        ax.set_ylim(0, min(fs/2, 8000))
    axes[2].set_xlabel('Time (s)')

    plt.tight_layout()
    plt.show()

# Show best, worst, and a median chunk
sorted_idx = np.argsort(all_mse)
show_indices = [
    sorted_idx[0],                        # best
    sorted_idx[len(sorted_idx)//2],        # median
    sorted_idx[-1],                        # worst
]

for rank, i in enumerate(show_indices):
    labels = ['Best', 'Median', 'Worst']
    plot_comparison(all_x[i], all_y[i], fs, title=f'{labels[rank]} — ', chunk_idx=i)

In [ ]:
# Listen to best and worst
best_i, worst_i = sorted_idx[0], sorted_idx[-1]

print(f'=== Best chunk (#{best_i}, MSE={all_mse[best_i]:.6f}) ===')
print('Input:')
ipd.display(ipd.Audio(all_x[best_i], rate=fs))
print('Output:')
ipd.display(ipd.Audio(all_y[best_i], rate=fs))

print(f'\n=== Worst chunk (#{worst_i}, MSE={all_mse[worst_i]:.6f}) ===')
print('Input:')
ipd.display(ipd.Audio(all_x[worst_i], rate=fs))
print('Output:')
ipd.display(ipd.Audio(all_y[worst_i], rate=fs))

## 4. Per-Chunk Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# MSE per chunk
axes[0].bar(range(len(all_mse)), all_mse, color='steelblue')
axes[0].axhline(np.mean(all_mse), color='red', linestyle='--', label=f'mean={np.mean(all_mse):.5f}')
axes[0].set_xlabel('Chunk index')
axes[0].set_ylabel('MSE')
axes[0].set_title('Reconstruction MSE per Chunk')
axes[0].legend()

# MSE histogram
axes[1].hist(all_mse, bins=15, color='steelblue', edgecolor='white')
axes[1].axvline(np.mean(all_mse), color='red', linestyle='--', label=f'mean={np.mean(all_mse):.5f}')
axes[1].set_xlabel('MSE')
axes[1].set_ylabel('Count')
axes[1].set_title('MSE Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary stats
print(f'MSE — mean: {np.mean(all_mse):.6f}, std: {np.std(all_mse):.6f}')
print(f'       min: {np.min(all_mse):.6f}, max: {np.max(all_mse):.6f}')
print(f'       median: {np.median(all_mse):.6f}')

## 5. Latent Space Visualization

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Use time-averaged mu as per-chunk latent representation
mu_avg = all_mu_cat.mean(dim=1).numpy()  # [N, latent_size]
N = mu_avg.shape[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA
pca = PCA(n_components=2)
mu_pca = pca.fit_transform(mu_avg)
sc = axes[0].scatter(mu_pca[:, 0], mu_pca[:, 1], c=range(N), cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
for i in range(N):
    axes[0].annotate(str(i), (mu_pca[i, 0], mu_pca[i, 1]), fontsize=7, ha='center', va='bottom')
axes[0].set_title(f'PCA of Latent Means (explained var: {pca.explained_variance_ratio_.sum():.1%})')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(sc, ax=axes[0], label='Chunk index')

# t-SNE
perplexity = min(5, N - 1)
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
mu_tsne = tsne.fit_transform(mu_avg)
sc2 = axes[1].scatter(mu_tsne[:, 0], mu_tsne[:, 1], c=range(N), cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
for i in range(N):
    axes[1].annotate(str(i), (mu_tsne[i, 0], mu_tsne[i, 1]), fontsize=7, ha='center', va='bottom')
axes[1].set_title('t-SNE of Latent Means')
axes[1].set_xlabel('Dim 1')
axes[1].set_ylabel('Dim 2')
plt.colorbar(sc2, ax=axes[1], label='Chunk index')

plt.tight_layout()
plt.show()

In [ ]:
# Latent dimension statistics
mu_flat = all_mu_cat.reshape(-1, all_mu_cat.shape[-1]).numpy()  # [N*T_ctl, latent_size]
latent_std = mu_flat.std(axis=0)
latent_mean = mu_flat.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Std per dimension (active vs collapsed)
sorted_std = np.sort(latent_std)[::-1]
axes[0].bar(range(len(sorted_std)), sorted_std, color='steelblue', width=1.0)
axes[0].axhline(0.1, color='red', linestyle='--', alpha=0.5, label='collapse threshold')
axes[0].set_xlabel('Latent dimension (sorted by std)')
axes[0].set_ylabel('Std of μ')
axes[0].set_title(f'Latent Activity — {(latent_std > 0.1).sum()} active dims (of {len(latent_std)})')
axes[0].legend()

# Mean per dimension
axes[1].bar(range(len(latent_mean)), latent_mean, color='coral', width=1.0)
axes[1].set_xlabel('Latent dimension')
axes[1].set_ylabel('Mean of μ')
axes[1].set_title('Latent Means')

plt.tight_layout()
plt.show()

print(f'Active latent dimensions (std > 0.1): {(latent_std > 0.1).sum()} / {len(latent_std)}')
print(f'Top 10 most active dims: {np.argsort(latent_std)[::-1][:10]}')

## 6. Latent Interpolation

In [ ]:
def interpolate_and_decode(ddsp, z_start, z_end, n_steps=8):
    """Spherical interpolation between two latent codes."""
    audios = []
    alphas = np.linspace(0, 1, n_steps)
    for alpha in alphas:
        # Slerp
        z_start_flat = z_start.reshape(-1)
        z_end_flat = z_end.reshape(-1)
        omega = torch.acos(torch.clamp(
            F.cosine_similarity(z_start_flat.unsqueeze(0), z_end_flat.unsqueeze(0)),
            -1 + 1e-7, 1 - 1e-7
        ))
        if omega.abs() < 1e-6:
            z_interp = (1 - alpha) * z_start + alpha * z_end
        else:
            z_interp = (torch.sin((1 - alpha) * omega) / torch.sin(omega)) * z_start + \
                       (torch.sin(alpha * omega) / torch.sin(omega)) * z_end

        z_interp = z_interp.unsqueeze(0).to(device)  # [1, T_ctl, latent]
        audio = decode_from_z(ddsp, z_interp)
        audios.append(audio.squeeze().cpu().numpy())
    return audios, alphas

# Pick two chunks to interpolate between
idx_a, idx_b = sorted_idx[0], sorted_idx[-1]  # best and worst reconstruction
print(f'Interpolating between chunk {idx_a} and chunk {idx_b}')

z_a = all_z_cat[idx_a]  # [T_ctl, latent_size]
z_b = all_z_cat[idx_b]

interp_audios, interp_alphas = interpolate_and_decode(ddsp, z_a, z_b, n_steps=8)

In [ ]:
# Plot interpolation spectrograms
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle(f'Latent Interpolation: Chunk {idx_a} → Chunk {idx_b}', fontsize=13)

for i, (audio, alpha) in enumerate(zip(interp_audios, interp_alphas)):
    ax = axes[i // 4][i % 4]
    ax.specgram(audio, Fs=fs, NFFT=2048, noverlap=1024, cmap='magma')
    ax.set_title(f'α={alpha:.2f}')
    ax.set_ylim(0, min(fs/2, 8000))
    if i % 4 == 0:
        ax.set_ylabel('Freq (Hz)')
    if i >= 4:
        ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

In [ ]:
# Listen to interpolation
print(f'Interpolation: chunk {idx_a} → chunk {idx_b}')
for i, (audio, alpha) in enumerate(zip(interp_audios, interp_alphas)):
    print(f'  α = {alpha:.2f}:')
    ipd.display(ipd.Audio(audio, rate=fs))

## 7. Random Latent Generation

In [ ]:
# Compute latent statistics from the dataset for informed sampling
mu_all = all_mu_cat.numpy()  # [N, T_ctl, latent_size]
global_mean = mu_all.mean(axis=(0, 1))  # [latent_size]
global_std = mu_all.std(axis=(0, 1))    # [latent_size]

T_ctl = all_mu_cat.shape[1]
latent_size = all_mu_cat.shape[2]

n_random = 6
random_audios = []

for i in range(n_random):
    # Sample from N(global_mean, global_std)
    z_rand = torch.from_numpy(
        np.random.randn(1, T_ctl, latent_size).astype(np.float32) * global_std + global_mean
    ).to(device)
    audio = decode_from_z(ddsp, z_rand)
    random_audios.append(audio.squeeze().cpu().numpy())

print(f'Generated {n_random} random samples from latent distribution')

In [ ]:
# Plot and listen to random generations
n_cols = 3
n_rows = (n_random + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
fig.suptitle('Random Latent Generations', fontsize=13)

for i in range(n_random):
    ax = axes[i // n_cols][i % n_cols] if n_rows > 1 else axes[i % n_cols]
    ax.specgram(random_audios[i], Fs=fs, NFFT=2048, noverlap=1024, cmap='magma')
    ax.set_title(f'Random #{i}')
    ax.set_ylim(0, min(fs/2, 8000))

# Hide unused axes
for i in range(n_random, n_rows * n_cols):
    ax = axes[i // n_cols][i % n_cols] if n_rows > 1 else axes[i % n_cols]
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Listen to random generations
for i, audio in enumerate(random_audios):
    print(f'Random #{i}:')
    ipd.display(ipd.Audio(audio, rate=fs))

## 8. Latent Dimension Manipulation

Sweep individual latent dimensions to see what each controls.

In [ ]:
# Pick the most active latent dimensions and a reference chunk
top_dims = np.argsort(latent_std)[::-1][:4]  # 4 most active dims
ref_chunk = sorted_idx[0]  # best reconstruction as reference
z_ref = all_mu_cat[ref_chunk].clone()  # [T_ctl, latent_size] — use mu (no noise)

sweep_range = np.linspace(-3, 3, 7)  # sweep from -3σ to +3σ

fig, axes = plt.subplots(len(top_dims), len(sweep_range), figsize=(18, 3 * len(top_dims)))
fig.suptitle(f'Latent Dimension Sweeps (reference: chunk {ref_chunk})', fontsize=13)

sweep_audios = {}  # (dim, sweep_val) → audio

for row, dim in enumerate(top_dims):
    dim_std = latent_std[dim]
    dim_mean = global_mean[dim]
    for col, sigma in enumerate(sweep_range):
        z_mod = z_ref.clone()
        z_mod[:, dim] = dim_mean + sigma * dim_std  # set all time steps
        z_mod_dev = z_mod.unsqueeze(0).to(device)
        audio = decode_from_z(ddsp, z_mod_dev).squeeze().cpu().numpy()
        sweep_audios[(dim, sigma)] = audio

        ax = axes[row][col]
        ax.specgram(audio, Fs=fs, NFFT=2048, noverlap=1024, cmap='magma')
        ax.set_ylim(0, min(fs/2, 8000))
        if row == 0:
            ax.set_title(f'{sigma:+.1f}σ')
        if col == 0:
            ax.set_ylabel(f'Dim {dim}\n(σ={dim_std:.3f})')
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# Listen to one dimension sweep
listen_dim = top_dims[0]
print(f'Sweeping latent dimension {listen_dim} (std={latent_std[listen_dim]:.3f}):')
for sigma in sweep_range:
    audio = sweep_audios[(listen_dim, sigma)]
    print(f'  {sigma:+.1f}σ:')
    ipd.display(ipd.Audio(audio, rate=fs))

## 9. Synth Parameter Visualization

Visualize the predicted sinusoidal parameters (frequencies, amplitudes).

In [ ]:
@torch.no_grad()
def get_synth_params(ddsp, audio, features):
    """Get raw synth parameters from the encoder→decoder pipeline."""
    if audio.dim() == 1:
        audio = audio.unsqueeze(0)
    if features.dim() == 2:
        features = features.unsqueeze(0)
    audio = audio.to(device)
    features = features.to(device)

    mu, logvar = ddsp.encoder(audio)
    z, _ = ddsp.encoder.reparametrize(mu, logvar)
    z = ddsp._smooth_latents(z)

    T_ctl = z.shape[1]
    feat = features if ddsp.feature_dim > 0 else torch.zeros(1, T_ctl, 1, device=device)
    T_min = min(feat.shape[1], z.shape[1])
    synth_params = ddsp.decoder(feat[:, :T_min], z[:, :T_min],
                                audio=audio, hop_size=ddsp.resampling_factor)
    return synth_params  # [1, n_params, T_ctl]

# Get params for the best chunk
ref_audio, ref_features, _ = dataset[ref_chunk]
params = get_synth_params(ddsp, ref_audio, ref_features)
params_np = params.squeeze(0).cpu().numpy()  # [n_params, T_ctl]

n_sines = params_np.shape[0] // 4
omega = params_np[:n_sines, :]            # [n_sines, T_ctl]
phi = params_np[n_sines:2*n_sines, :]
amp_s = params_np[2*n_sines:3*n_sines, :]
amp_e = params_np[3*n_sines:, :]

# Convert omega to Hz
freqs_hz = np.abs(omega) * fs / (2 * np.pi)
# Amplitudes via softplus
amp_mean = (np.log1p(np.exp(amp_s)) + np.log1p(np.exp(amp_e))) / (2 * n_sines)

print(f'Synth params shape: {params_np.shape} ({n_sines} sines × 4 params)')
print(f'Frequency range: {freqs_hz.min():.1f} – {freqs_hz.max():.1f} Hz')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Frequency tracks
t_frames = np.arange(omega.shape[1]) * resampling_factor / fs
for s in range(n_sines):
    amp_scale = amp_mean[s, :]
    amp_norm = amp_scale / (amp_scale.max() + 1e-8)
    axes[0].scatter(t_frames, freqs_hz[s, :], s=1, c=plt.cm.viridis(amp_norm), alpha=0.6)
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_title(f'Sinusoidal Frequency Tracks — Chunk {ref_chunk} ({n_sines} sines)')
axes[0].set_ylim(0, min(freqs_hz.max() * 1.1, fs/2))

# Amplitude heatmap
# Sort sines by mean frequency for cleaner visualization
sort_order = np.argsort(freqs_hz.mean(axis=1))
im = axes[1].imshow(amp_mean[sort_order, :], aspect='auto', origin='lower', cmap='magma',
                     extent=[0, t_frames[-1], 0, n_sines])
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Sine index (sorted by freq)')
axes[1].set_title('Amplitude (softplus, sorted by frequency)')
plt.colorbar(im, ax=axes[1], label='Amplitude')

plt.tight_layout()
plt.show()